In [ ]:
# Challenge 1 - Real-Time Weather Alerts Agent
# Install the dependencies required for ADK, HTTP tools, and an optional LiteLLM provider.
%pip install -q --upgrade google-adk litellm requests googlemaps

import os
import json
import logging
from typing import Any, Dict, List, Optional

import requests
import vertexai
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

PROJECT_ID = "qwiklabs-gcp-02-9e12deb8c42f"
LOCATION = "us-central1"
MODEL_GEMINI = "gemini-2.0-flash"

vertexai.init(project=PROJECT_ID, location=LOCATION)
logging.basicConfig(level=logging.INFO)
print(f"Vertex AI initialized for {PROJECT_ID} in {LOCATION}")

Vertex AI initialized for qwiklabs-gcp-02-9e12deb8c42f in us-central1


In [8]:
from getpass import getpass# Compatible agent-level smoke test
from vertexai.preview import reasoning_engines

app = reasoning_engines.AdkApp(agent=weather_agent_gemini)
session = app.create_session(user_id="challenge-one-tester")

for event in app.stream_query(
    user_id="challenge-one-tester",
    session_id=session["id"],
    message="Give me a concise weather alert for Miami, Florida.",
):
    last_event = event

print(last_event["content"]["parts"][0]["text"])

GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")
if not GOOGLE_MAPS_API_KEY:
    GOOGLE_MAPS_API_KEY = getpass("Paste your Google Maps API key: ")
    os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY


def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Convert a United States location to latitude and longitude with Google Maps Geocoding API.

    Args:
        location: A city, state, or complete U.S. address.

    Returns:
        A dictionary with latitude, longitude, and formatted_address, or None when no result is found.
    """
    if not GOOGLE_MAPS_API_KEY:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY before using Google Maps Geocoding.")
    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json",
        params={"address": location, "components": "country:US", "key": GOOGLE_MAPS_API_KEY},
        timeout=20,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("status") != "OK" or not payload.get("results"):
        return None
    result = payload["results"][0]
    coordinates = result["geometry"]["location"]
    return {
        "latitude": float(coordinates["lat"]),
        "longitude": float(coordinates["lng"]),
        "formatted_address": result["formatted_address"],
    }


def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Retrieve the National Weather Service forecast for a U.S. latitude and longitude.

    Args:
        lat: Latitude of the U.S. location.
        lon: Longitude of the U.S. location.

    Returns:
        Up to six forecast periods containing name, temperature, wind, short forecast, and detail.
    """
    point_response = requests.get(
        f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS, timeout=20
    )
    point_response.raise_for_status()
    forecast_url = point_response.json()["properties"]["forecast"]
    forecast_response = requests.get(forecast_url, headers=NWS_HEADERS, timeout=20)
    forecast_response.raise_for_status()
    periods = forecast_response.json()["properties"]["periods"]
    return [
        {
            "period": period["name"],
            "temperature": f"{period['temperature']} {period['temperatureUnit']}",
            "wind": f"{period['windSpeed']} {period['windDirection']}",
            "forecast": period["shortForecast"],
            "detail": period["detailedForecast"],
        }
        for period in periods[:6]
    ]


WEATHER_AGENT_INSTRUCTIONS = """You are Pat, a careful U.S. real-time weather-alert agent.
Use get_lat_lon for a location, then get_extended_weather_forecast for its coordinates.
Summarize the near-term forecast clearly. Highlight hazards such as severe storms, extreme heat,
heavy snow, high winds, flooding, or fire weather. Do not invent weather information. Explain that
NWS only covers U.S. locations and ask for a U.S. city/state if needed."""

weather_agent_gemini = LlmAgent(
    name="pat_weather_gemini",
    model=MODEL_GEMINI,
    description="Provides current National Weather Service forecasts and weather alerts for U.S. locations.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

# The same agent design can use an approved third-party model after its API key is set in the runtime.
weather_agent_openai = LlmAgent(
    name="pat_weather_openai",
    model=LiteLlm(model="openai/gpt-4o-mini"),
    description="Third-party-model version of the U.S. weather agent.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

print("Created Gemini and LiteLLM weather-agent definitions.")

/usr/local/lib/python3.12/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
ERROR:google_adk.google.adk.workflow._node_runner:Node execution failed with exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "/usr/local/lib/python3.12/dist-packages/google/adk/workflow/_base_node.py", line 166, in run
    async for item in agen:
  File "/usr/local/lib/python3.12/dist-packages/google/adk/agents/llm_agent.py", line 614, in

KeyError: 'content'

In [9]:
# Use the current Flash model available to this Vertex AI lab project.
MODEL_GEMINI = "gemini-2.5-flash"
weather_agent_gemini = LlmAgent(
    name="pat_weather_gemini",
    model=MODEL_GEMINI,
    description="Provides current National Weather Service forecasts and weather alerts for U.S. locations.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

app = reasoning_engines.AdkApp(agent=weather_agent_gemini)
session = app.create_session(user_id="challenge-one-tester")
for event in app.stream_query(
    user_id="challenge-one-tester",
    session_id=session["id"],
    message="Give me a concise weather alert for Miami, Florida.",
):
    last_event = event

print(last_event["content"]["parts"][0]["text"])

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


**Weather Alert for Miami, Florida:**

**Extreme Heat Advisory:** Heat index values will be dangerously high. Expect heat index values as high as **110°F today**, **105°F tonight**, and **106°F on Friday**.

There is a **chance of showers and thunderstorms** each afternoon and evening through Saturday. Today's high will be near 91°F, with a low tonight around 84°F. Friday's high will be near 90°F.


In [ ]:
# Tests required for Challenge 1: demonstrate the agent for multiple U.S. cities.
TEST_LOCATIONS = ["New York, NY", "Miami, FL", "Denver, CO"]


def test_weather_tools(location: str) -> None:
    coordinates = get_lat_lon(location)
    assert coordinates is not None, f"No coordinates returned for {location}"
    forecast = get_extended_weather_forecast(coordinates["latitude"], coordinates["longitude"])
    assert forecast, f"No NWS forecast returned for {location}"
    first_period = forecast[0]
    print(f"\n{coordinates['formatted_address']}")
    print(json.dumps(first_period, indent=2))


for test_location in TEST_LOCATIONS:
    test_weather_tools(test_location)

# Optional agent-level smoke test after the tool tests pass.
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(agent=weather_agent_gemini, enable_tracing=False)
session = app.create_session(user_id="challenge-one-tester")
for event in app.stream_query(
    user_id="challenge-one-tester",
    session_id=session.id,
    message="Give me a concise weather alert for Miami, Florida.",
):
    last_event = event
print(last_event["content"]["parts"][0]["text"])